In [9]:
import pandas as pd
import numpy as np

print("🐕PET DATA CLEANER STARTED...")

# Step 1: File load karo
file_name = input(" /content/pet.csv ")

try:
    df = pd.read_csv(file_name)
    print(" File load ho gaya!")

    # Step 2: Basic info batayo
    print(f" Total rows: {len(df)}")
    print(f" Total columns: {len(df.columns)}")

    # Step 3: Missing values fix karo
    missing_total = df.isnull().sum().sum()
    if missing_total > 0:
        print(f" {missing_total} missing values fix kiye...")

        # Number columns ke liye
        number_cols = df.select_dtypes(include=[np.number]).columns
        for col in number_cols:
            if df[col].isnull().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)

        # Text columns ke liye
        text_cols = df.select_dtypes(include=['object']).columns
        for col in text_cols:
            if df[col].isnull().sum() > 0:
                df[col].fillna('Unknown', inplace=True)

    # Step 4: Duplicates hatao
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"{duplicates} duplicate rows hataye...")
        df = df.drop_duplicates()

    # Step 5: Text clean karo
    print(" Text columns clean kar raha hoon...")
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip().str.title()

    # Step 6: Outliers handle karo
    print(" Outliers handle kar raha hoon...")
    number_cols = df.select_dtypes(include=[np.number]).columns

    for col in number_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        # Outliers ko cap karo
        df[col] = np.where(df[col] < lower, lower, df[col])
        df[col] = np.where(df[col] > upper, upper, df[col])

    # Step 7: Save karo
    output_file = file_name.replace('.csv', '_clean.csv')
    df.to_csv(output_file, index=False)

    # Final report
    print("\n CLEANING COMPLETE!")
    print(f"Cleaned file: {output_file}")
    print(f" Final rows: {len(df)}")
    print(f" Final columns: {len(df.columns)}")

except Exception as e:
    print(f" Error: {e}")
    print(" File load nahi hui. Check karo:")
    print("   - File name sahi hai?")
    print("   - File same folder mein hai?")

🐕PET DATA CLEANER STARTED...
 /content/pet.csv pet.csv
 File load ho gaya!
 Total rows: 5000
 Total columns: 10
 Text columns clean kar raha hoon...
 Outliers handle kar raha hoon...

 CLEANING COMPLETE!
Cleaned file: pet_clean.csv
 Final rows: 5000
 Final columns: 10


In [10]:
import pandas as pd
import numpy as np

def quick_data_quality_check(file_path):
    """Quick data quality assessment"""
    df = pd.read_csv('/content/pet_clean.csv')

    print("=== QUICK DATA QUALITY CHECK ===")
    print(f"Dataset: {file_path}")
    print(f"Shape: {df.shape}")

    # Basic info
    print("\n--- BASIC INFO ---")
    print(f"Columns: {list(df.columns)}")
    print(f"Data types:\n{df.dtypes}")

    # Missing values
    print("\n--- MISSING VALUES ---")
    missing = df.isnull().sum()
    missing_percent = (missing / len(df)) * 100
    missing_report = pd.DataFrame({
        'Missing Count': missing,
        'Missing %': missing_percent
    })
    print(missing_report[missing_report['Missing Count'] > 0])

    # Duplicates
    print(f"\n--- DUPLICATES ---")
    print(f"Duplicate rows: {df.duplicated().sum()}")

    # Numerical data summary
    print("\n--- NUMERICAL DATA SUMMARY ---")
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    if len(numerical_cols) > 0:
        print(df[numerical_cols].describe())

    # Categorical data summary
    print("\n--- CATEGORICAL DATA SUMMARY ---")
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        print(f"\n{col}:")
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Most common: {df[col].mode().iloc[0] if len(df[col].mode()) > 0 else 'N/A'}")

    # Data quality issues
    print("\n--- DATA QUALITY ISSUES ---")
    issues = []

    # Check for negative values in numerical columns
    for col in numerical_cols:
        if (df[col] < 0).any():
            issues.append(f"Negative values found in {col}")

    # Check for zeros in weight and age
    if 'Weight_kg' in df.columns and (df['Weight_kg'] <= 0).any():
        issues.append("Zero or negative weight values found")

    if 'Age_Years' in df.columns and (df['Age_Years'] < 0).any():
        issues.append("Negative age values found")

    if issues:
        for issue in issues:
            print(f"⚠️  {issue}")
    else:
        print("✓ No major data quality issues found")

    return df

# Run quick check
if __name__ == "__main__":
    df = quick_data_quality_check('pet_health_complete_dataset.csv')

=== QUICK DATA QUALITY CHECK ===
Dataset: pet_health_complete_dataset.csv
Shape: (5000, 10)

--- BASIC INFO ---
Columns: ['Pet_Name', 'Breed', 'Pet_Type', 'Age_Years', 'Weight_kg', 'Temperature_C', 'Heartbeat_BPM', 'Health_Status', 'Current_Behavior', 'Recommendation']
Data types:
Pet_Name             object
Breed                object
Pet_Type             object
Age_Years           float64
Weight_kg           float64
Temperature_C       float64
Heartbeat_BPM       float64
Health_Status        object
Current_Behavior     object
Recommendation       object
dtype: object

--- MISSING VALUES ---
Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []

--- DUPLICATES ---
Duplicate rows: 0

--- NUMERICAL DATA SUMMARY ---
         Age_Years    Weight_kg  Temperature_C  Heartbeat_BPM
count  5000.000000  5000.000000    5000.000000    5000.000000
mean      7.612600    16.170700      38.770716     126.787700
std       4.619003    12.261591       0.618277      29.414527
min       1.000000  

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

In [13]:
df = pd.read_csv('/content/pet_clean.csv')
print(" Clean data load ho gaya!")

 Clean data load ho gaya!


In [14]:
df['Breed_Code'] = df['Breed'].astype('category').cat.codes


In [15]:
X = df[['Breed_Code', 'Age_Years', 'Weight_kg', 'Temperature_C', 'Heartbeat_BPM']]
y = df['Health_Status']

print(f" Training on {len(X)} samples")

 Training on 5000 samples


In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [18]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)


In [24]:
print(f"\n DETAILED CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print(f"\n CONFUSION MATRIX:")
cm = confusion_matrix(y_test, y_pred)
print(cm)



📊 DETAILED CLASSIFICATION REPORT:
              precision    recall  f1-score   support

       Alert       0.80      0.86      0.83       187
    Critical       0.77      0.69      0.73       118
      Normal       1.00      1.00      1.00       695

    accuracy                           0.93      1000
   macro avg       0.86      0.85      0.85      1000
weighted avg       0.93      0.93      0.93      1000


🔄 CONFUSION MATRIX:
[[160  25   2]
 [ 36  82   0]
 [  3   0 692]]


In [34]:
joblib.dump(model, 'pet_health_model.pkl')
print(f"\n Model saved as 'pet_health_model.pkl'")


 Model saved as 'pet_health_model.pkl'


AB HAM PREDICT KARANGA